# McNemar's Test — Statistical Comparison of Models

When two models are trained on different data strategies (SMOTE vs class weights) and evaluated
on the same test set, we need a statistical test to determine whether their performance difference
is significant or just noise.

**McNemar's test** is designed for exactly this: comparing two classifiers on paired data.
It focuses on the samples where the two models *disagree* and tests whether the disagreements
are symmetric.

### How it works

Build a 2x2 contingency table of correct/incorrect predictions:

|  | Model B correct | Model B wrong |
|---|---|---|
| **Model A correct** | a (both right) | b (only A right) |
| **Model A wrong** | c (only B right) | d (both wrong) |

McNemar's only looks at cells **b** and **c** (the disagreements). If b and c are roughly equal,
the models perform equivalently. The test statistic is:

$$\chi^2 = \frac{(b - c)^2}{b + c}$$

Use the exact binomial version when b + c < 25.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
import os
from statsmodels.stats.contingency_tables import mcnemar

DATA_DIR = '../data'

## 1. Load Test Data and Both Models

In [ ]:
# Test data is the same for both — only the training strategy differed
X_test = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
y_test = np.load(os.path.join(DATA_DIR, 'y_test.npy'))

print(f'Test set: {len(y_test):,} samples ({y_test.sum():.0f} fraud, {(y_test==0).sum():,} legit)')

# Load both XGBoost models
model_smote = joblib.load('../models/train_smote/XGBoost.joblib')
model_no_smote = joblib.load('../models/train_no_smote/XGBoost.joblib')

print(f'Model A: XGBoost trained with SMOTE')
print(f'Model B: XGBoost trained with class_weight=balanced')

In [ ]:
# Get probability scores
scores_smote = model_smote.predict_proba(X_test)[:, 1]
scores_no_smote = model_no_smote.predict_proba(X_test)[:, 1]

## 2. Helper Functions

In [ ]:
def build_mcnemar_table(y_true, y_pred_a, y_pred_b):
    """
    Build the 2x2 contingency table for McNemar's test.
    
    Returns:
        table: 2x2 numpy array
        info: dict with cell counts and labels
    """
    correct_a = (y_pred_a == y_true)
    correct_b = (y_pred_b == y_true)
    
    a = (correct_a & correct_b).sum()       # both correct
    b = (correct_a & ~correct_b).sum()      # only A correct
    c = (~correct_a & correct_b).sum()      # only B correct
    d = (~correct_a & ~correct_b).sum()     # both wrong
    
    table = np.array([[a, b], [c, d]])
    
    info = {
        'both_correct': int(a),
        'only_A_correct': int(b),
        'only_B_correct': int(c),
        'both_wrong': int(d),
        'total_disagreements': int(b + c),
    }
    
    return table, info


def run_mcnemar(y_true, y_pred_a, y_pred_b, name_a='Model A', name_b='Model B', alpha=0.05):
    """
    Run McNemar's test and print a full report.
    
    Uses exact binomial test when disagreements < 25,
    chi-squared with continuity correction otherwise.
    """
    table, info = build_mcnemar_table(y_true, y_pred_a, y_pred_b)
    
    print(f'=== McNemar\'s Test: {name_a} vs {name_b} ===')
    print(f'  Samples evaluated: {len(y_true):,}')
    print()
    
    # Print contingency table
    print(f'  Contingency Table:')
    print(f'  {"":>20s} | {name_b+" correct":>15s} | {name_b+" wrong":>15s}')
    print(f'  {"-"*20}-+-{"-"*15}-+-{"-"*15}')
    print(f'  {name_a+" correct":>20s} | {info["both_correct"]:>15,} | {info["only_A_correct"]:>15,}')
    print(f'  {name_a+" wrong":>20s} | {info["only_B_correct"]:>15,} | {info["both_wrong"]:>15,}')
    print()
    
    # Disagreement analysis
    b = info['only_A_correct']
    c = info['only_B_correct']
    print(f'  Disagreements: {b + c:,} ({(b+c)/len(y_true):.2%} of samples)')
    print(f'    Only {name_a} correct: {b:,}')
    print(f'    Only {name_b} correct: {c:,}')
    print()
    
    # Choose exact or chi-squared
    use_exact = (b + c) < 25
    if use_exact:
        print(f'  Using: Exact binomial test (disagreements < 25)')
    else:
        print(f'  Using: Chi-squared with continuity correction')
    
    result = mcnemar(table, exact=use_exact, correction=True)
    
    print(f'  Statistic: {result.statistic:.4f}')
    print(f'  P-value:   {result.pvalue:.6f}')
    print()
    
    if result.pvalue < alpha:
        winner = name_a if b > c else name_b
        print(f'  Result: SIGNIFICANT difference (p < {alpha})')
        print(f'  {winner} performs significantly better.')
    else:
        print(f'  Result: NO significant difference (p >= {alpha})')
        print(f'  The two models perform equivalently on this test set.')
    
    return result, info

## 3. Test on ALL Samples

This tells us whether the two training strategies produce different overall classification behavior.
Since 99.8% of samples are legitimate, this test is dominated by performance on the majority class.

In [ ]:
# Use a common threshold — e.g., 0.5 or a tuned one
THRESHOLD = 0.5

y_pred_smote = (scores_smote >= THRESHOLD).astype(int)
y_pred_no_smote = (scores_no_smote >= THRESHOLD).astype(int)

print(f'Threshold: {THRESHOLD}\n')
result_all, info_all = run_mcnemar(
    y_test, y_pred_smote, y_pred_no_smote,
    name_a='XGB_SMOTE', name_b='XGB_NoSMOTE'
)

### Test at multiple thresholds

The comparison can change depending on the operating point. Let's sweep thresholds.

In [ ]:
thresholds_to_test = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
results_summary = []

for t in thresholds_to_test:
    y_pred_a = (scores_smote >= t).astype(int)
    y_pred_b = (scores_no_smote >= t).astype(int)
    table, info = build_mcnemar_table(y_test, y_pred_a, y_pred_b)
    
    use_exact = info['total_disagreements'] < 25
    result = mcnemar(table, exact=use_exact, correction=True)
    
    b = info['only_A_correct']
    c = info['only_B_correct']
    
    results_summary.append({
        'threshold': t,
        'only_SMOTE_correct': b,
        'only_NoSMOTE_correct': c,
        'total_disagreements': b + c,
        'p_value': result.pvalue,
        'significant': result.pvalue < 0.05,
        'better_model': 'SMOTE' if b > c else ('NoSMOTE' if c > b else 'Tied'),
    })

df_results = pd.DataFrame(results_summary)
print('=== McNemar\'s Test Across Thresholds (All Samples) ===\n')
print(df_results.to_string(index=False))

## 4. Test on FRAUD Samples Only

The overall test is dominated by the majority class. For fraud detection, what we really care about
is: **do the two models differ in their ability to catch actual fraud?**

We filter to only the fraud cases and re-run. Here "correct" means the model flagged the fraud,
and "wrong" means it missed it.

**Caveat:** With only ~98 fraud samples in the test set, we'll have very few disagreements,
so we must use the exact binomial version of McNemar's test.

In [ ]:
# Filter to fraud only
fraud_mask = y_test == 1
X_test_fraud = X_test[fraud_mask]
y_test_fraud = y_test[fraud_mask]
scores_smote_fraud = scores_smote[fraud_mask]
scores_no_smote_fraud = scores_no_smote[fraud_mask]

print(f'Fraud samples in test set: {fraud_mask.sum()}')
print(f'(This is our sample size — expect low power)\n')

y_pred_smote_fraud = (scores_smote_fraud >= THRESHOLD).astype(int)
y_pred_no_smote_fraud = (scores_no_smote_fraud >= THRESHOLD).astype(int)

print(f'Threshold: {THRESHOLD}')
print(f'SMOTE recall:    {y_pred_smote_fraud.sum()}/{len(y_test_fraud)} = {y_pred_smote_fraud.mean():.4f}')
print(f'NoSMOTE recall:  {y_pred_no_smote_fraud.sum()}/{len(y_test_fraud)} = {y_pred_no_smote_fraud.mean():.4f}\n')

result_fraud, info_fraud = run_mcnemar(
    y_test_fraud, y_pred_smote_fraud, y_pred_no_smote_fraud,
    name_a='XGB_SMOTE', name_b='XGB_NoSMOTE'
)

In [ ]:
# Fraud-only test across thresholds
fraud_results = []

for t in thresholds_to_test:
    y_pred_a = (scores_smote_fraud >= t).astype(int)
    y_pred_b = (scores_no_smote_fraud >= t).astype(int)
    table, info = build_mcnemar_table(y_test_fraud, y_pred_a, y_pred_b)
    
    b = info['only_A_correct']
    c = info['only_B_correct']
    
    # Always use exact test — small sample
    if (b + c) == 0:
        p_val = 1.0
    else:
        result = mcnemar(table, exact=True)
        p_val = result.pvalue
    
    fraud_results.append({
        'threshold': t,
        'SMOTE_recall': y_pred_a.mean(),
        'NoSMOTE_recall': y_pred_b.mean(),
        'only_SMOTE_caught': b,
        'only_NoSMOTE_caught': c,
        'disagreements': b + c,
        'p_value': p_val,
        'significant': p_val < 0.05,
    })

df_fraud = pd.DataFrame(fraud_results)
print('=== McNemar\'s Test Across Thresholds (Fraud Only) ===\n')
print(df_fraud.to_string(index=False))

## 5. Test on LEGITIMATE Samples Only

For completeness: do the models differ in how many false alarms they produce?
Here "correct" means the model correctly let the legitimate transaction through.

In [ ]:
legit_mask = y_test == 0
scores_smote_legit = scores_smote[legit_mask]
scores_no_smote_legit = scores_no_smote[legit_mask]
y_test_legit = y_test[legit_mask]

print(f'Legitimate samples in test set: {legit_mask.sum():,}\n')

y_pred_smote_legit = (scores_smote_legit >= THRESHOLD).astype(int)
y_pred_no_smote_legit = (scores_no_smote_legit >= THRESHOLD).astype(int)

print(f'Threshold: {THRESHOLD}')
print(f'SMOTE false positive rate:    {y_pred_smote_legit.mean():.6f}')
print(f'NoSMOTE false positive rate:  {y_pred_no_smote_legit.mean():.6f}\n')

result_legit, info_legit = run_mcnemar(
    y_test_legit, y_pred_smote_legit, y_pred_no_smote_legit,
    name_a='XGB_SMOTE', name_b='XGB_NoSMOTE'
)

## 6. Compare ALL Model Pairs

We can also compare across model types. Let's run McNemar's for every pair
within the SMOTE-trained models (LogReg vs RF, RF vs XGB, LogReg vs XGB).

In [ ]:
from itertools import combinations

# Load all SMOTE-trained models
model_names = ['LogisticRegression', 'RandomForest', 'XGBoost']
models = {}
for name in model_names:
    path = f'../models/train_smote/{name}.joblib'
    if os.path.exists(path):
        models[name] = joblib.load(path)
        print(f'Loaded: {name}')
    else:
        print(f'Not found: {path}')

# Get predictions for each
predictions = {}
for name, model in models.items():
    scores = model.predict_proba(X_test)[:, 1]
    predictions[name] = (scores >= THRESHOLD).astype(int)

In [ ]:
# Pairwise McNemar's tests
print(f'=== Pairwise McNemar\'s Tests (threshold={THRESHOLD}) ===\n')

pair_results = []
for name_a, name_b in combinations(predictions.keys(), 2):
    table, info = build_mcnemar_table(y_test, predictions[name_a], predictions[name_b])
    use_exact = info['total_disagreements'] < 25
    result = mcnemar(table, exact=use_exact, correction=True)
    
    b = info['only_A_correct']
    c = info['only_B_correct']
    
    pair_results.append({
        'Model A': name_a,
        'Model B': name_b,
        'only_A_correct': b,
        'only_B_correct': c,
        'p_value': result.pvalue,
        'significant': result.pvalue < 0.05,
        'better': name_a if b > c else (name_b if c > b else 'Tied'),
    })

df_pairs = pd.DataFrame(pair_results)
print(df_pairs.to_string(index=False))

In [ ]:
# Same pairwise test but on fraud subset only
print(f'=== Pairwise McNemar\'s Tests — FRAUD ONLY (threshold={THRESHOLD}) ===\n')

pair_fraud_results = []
for name_a, name_b in combinations(predictions.keys(), 2):
    pred_a_fraud = predictions[name_a][fraud_mask]
    pred_b_fraud = predictions[name_b][fraud_mask]
    
    table, info = build_mcnemar_table(y_test_fraud, pred_a_fraud, pred_b_fraud)
    b = info['only_A_correct']
    c = info['only_B_correct']
    
    if (b + c) == 0:
        p_val = 1.0
    else:
        result = mcnemar(table, exact=True)
        p_val = result.pvalue
    
    pair_fraud_results.append({
        'Model A': name_a,
        'Model B': name_b,
        'A_recall': pred_a_fraud.mean(),
        'B_recall': pred_b_fraud.mean(),
        'only_A_caught': b,
        'only_B_caught': c,
        'p_value': p_val,
        'significant': p_val < 0.05,
    })

df_pair_fraud = pd.DataFrame(pair_fraud_results)
print(df_pair_fraud.to_string(index=False))

## 7. Interview Talking Points

### Why McNemar's and not a t-test on accuracy?
A t-test would require independent samples. Here both models are evaluated on the *same* test set,
so the predictions are paired. McNemar's is the correct test for comparing paired classifiers.

### Why not just compare AUPRC/AUROC directly?
A difference in AUPRC tells you the models rank differently, but doesn't tell you if that difference
is statistically significant. With a small fraud class (~98 test samples), even a 2-point AUPRC gap
might not be meaningful. McNemar's gives you a p-value.

### The fraud-only caveat
Running McNemar's on just the fraud subset (~98 samples) gives very few disagreements and low
statistical power. A non-significant result doesn't mean the models are equal — it may just mean
we don't have enough fraud examples to detect a real difference. In production, you'd accumulate
more labeled fraud cases over time and re-test.

### Exact vs chi-squared
When the number of disagreements (b + c) is small (< 25), the chi-squared approximation is unreliable.
Use `exact=True` for the exact binomial test. For large disagreement counts, the chi-squared version
with continuity correction (`correction=True`) is computationally cheaper and equally valid.

### Multiple comparisons
When testing multiple model pairs, apply Bonferroni correction: divide alpha by the number of tests.
For 3 pairwise tests, use alpha = 0.05/3 = 0.017 instead of 0.05.